# Capítulo 7 — Ajuste fino (finetuning) sem GPU

## Missão
Entender o ajuste fino pela **matemática que decide se ele cabe no seu bolso**. Você vai calcular memória de inferência e de treino, ver os formatos numéricos e a quantização na prática, contar parâmetros de LoRA, **treinar um LoRA de verdade em NumPy**, fundir modelos e preparar um conjunto de treino no formato que as APIs de ajuste fino aceitam. Antes disso, uma regra para decidir entre RAG e ajuste fino.

**No fim deste caderno você terá:** um LoRA treinado à mão (você vê a perda cair com 3% dos parâmetros) e o arquivo `saidas/treino.jsonl`, pronto para um ajuste fino real.

**Leitura essencial (opcional, livro em inglês):**
- When to Finetune (Finetuning and RAG) · PDF p. 335
- Memory Math · PDF p. 346
- Numerical Representations · PDF p. 349
- Quantization · PDF p. 352
- Parameter-Efficient Finetuning (LoRA) · PDF p. 356
- Model Merging · PDF p. 371

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap07_ajuste_fino_finetuning_sem_gpu.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
Ajuste fino muda os **pesos** do modelo. RAG muda o **contexto**.
- Use RAG quando falta **informação** (fatos, documentos que mudam).
- Use ajuste fino quando falta **comportamento**: formato, estilo, uma tarefa estreita repetida milhões de vezes, ou um modelo menor e mais barato fazendo o que um grande faz.

O gargalo do ajuste fino é a **memória**, e é por isso que existem LoRA e quantização.

### 7.1 **RAG ou ajuste fino?** Escreva `recomendar(caso)` com regras simples a partir de um dict com `falta` (`"informacao"` ou `"comportamento"`), `muda_rapido` (bool) e `volume_alto` (bool). Aplique a 4 cenários da Nimbus e explique cada resultado.

### 7.2 **Memória para inferência.** Os pesos ocupam `parâmetros × bytes por parâmetro`. Monte uma tabela em GB para modelos de 7, 13 e 70 bilhões de parâmetros em fp32 (4 bytes), fp16/bf16 (2), int8 (1) e int4 (0,5). Qual deles cabe numa GPU de 24 GB?

### 7.3 **Memória para treino.** No treino completo com Adam em fp16, uma conta aproximada por parâmetro treinável é: pesos (2 bytes) + gradientes (2) + estados do Adam (2 × 4 bytes, em fp32) = 12 bytes, sem contar as ativações. Calcule para um modelo de 7B com ajuste completo e com LoRA (só ~0,5% dos parâmetros treináveis; o resto fica congelado, só os 2 bytes do peso).

### 7.4 **Formatos numéricos.** Mostre na prática:
1. `0.1 + 0.2` em float32 e float16;
2. que 70.000 não cabe em float16 (vira `inf`);
3. que o bfloat16 aguenta o número grande, mas perde precisão.

Para simular o bfloat16, zere os 16 bits de baixo de um float32 (`view(np.uint32) & 0xFFFF0000`).

### 7.5 **Quantização int8.** Gere uma matriz de pesos 256×256 (`rng.normal(0, 0.02)`) com algumas colunas de escala maior (multiplique 8 colunas por 10). Quantize para int8:
- (a) **por tensor**: uma escala para tudo, `escala = max|W| / 127`;
- (b) **por coluna**: uma escala por coluna.

Desquantize e compare o erro médio absoluto. Por que o (b) ganha?

### 7.6 **Quantos parâmetros o LoRA treina?** Uma camada de 4096×4096 tem `d × k` pesos. O LoRA troca a atualização por `B (d×r) @ A (r×k)`, ou seja, `r × (d + k)` parâmetros. Calcule para r = 4, 8, 16 e 64 e a fração em relação à camada inteira.

### 7.7 **Treine um LoRA em NumPy.** Monte o experimento:
- um "modelo base" linear congelado `W0` (64×64);
- uma tarefa nova cujo modelo ideal é `W0 + ΔW`, com ΔW de posto 2;
- os dados: 512 exemplos `X` e `Y = X @ (W0 + ΔW).T`.

Treine só `A` (2×64, aleatória pequena) e `B` (64×2, zeros) com gradiente descendente na perda MSE de `X @ (W0 + B@A).T`, por 300 passos (lr=0.003). Mostre a perda a cada 50 passos e quantos parâmetros foram treinados em comparação com `W0`. Repare no platô do começo: como `B` começa em zero, o LoRA parte exatamente do modelo base e só depois "descobre" a direção da mudança.

### 7.8 **Fundir modelos (vetores de tarefa).** Com regressão linear simples: um modelo base `θ0` e duas tarefas, cada uma com o seu modelo ajustado (`θA`, `θB`, por mínimos quadrados a partir de `θ0` com poucos dados). Os **vetores de tarefa** são `θA - θ0` e `θB - θ0`. Compare o erro nas duas tarefas para: `θ0`, `θA`, `θB`, média simples `(θA+θB)/2` e soma de vetores de tarefa `θ0 + (θA-θ0) + (θB-θ0)`.

### 7.9 **Dados para um ajuste fino real.** Transforme os `TICKETS` no formato de conversa das APIs de ajuste fino: uma linha JSON por exemplo, `{"messages": [sistema, usuário, assistente]}`, em que o assistente responde só a categoria. Valide (3 mensagens, papéis certos, conteúdo não vazio), separe 80/20 em treino/validação com semente fixa e grave `../saidas/treino.jsonl` e `../saidas/validacao.jsonl`. Mostre o total de tokens.

## E para treinar de verdade?
- **Pela API** (o mais simples): o OpenAI e outros provedores aceitam esse `.jsonl` e cobram por token de treino. Comece com 50 a 100 exemplos bons, avalie com o pipeline do capítulo 4 e só então aumente.
- **Num modelo aberto**: bibliotecas como Hugging Face PEFT/TRL ou Unsloth treinam LoRA/QLoRA numa GPU de consumidor (ou no Google Colab). O LoRA que você treinou em NumPy é exatamente a mesma ideia, só que dentro de cada camada de atenção.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
